# A02 · Antireflection film

Check the reflection minimum and complex transmission against an independent thin-film calculation.

Run the solver cells separately or use **Run All** with the project environment. Each cell uses the same
geometry, excitation and quick sweep as the CLI. The native [Lumerical model](model.fsp) and
[reconstruction script](model.lsf) represent quick-sweep point 0. See the [case page](README.md)
for the published figures and [case.yaml](case.yaml) for the numerical inputs.

## Saved quick-sweep preview

![Reflection, transmission and phase](figures/spectrum-0.png)

![Pairwise solver differences](figures/solver-differences.png)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("A02")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## Analytic multilayer reference

In [ ]:
from rcwa_cv.references.multilayer import solve as solve_reference

results["reference"] = solve_sweep(solve_reference, samples, settings)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare the solvers

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a point shared by the solvers. The heatmaps show maximum discrepancies
over the tested points. Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)